# Deep Feature Quality Comparison

Load the test-run representations produced by `gz_foundation_models.py`, score each
encoder with the 2PCF score, intrinsic dimensionality (ID) score, clustering
accuracy, and Davies-Bouldin score (implemented in `backbone/`), then plot each
metric against the others in paper-quality PNGs saved to `../plots`.

In [ ]:
import os
import sys
import glob
import string

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append("..")
import backbone.custom_metrics.AstroMLmod3 as AstroMLmod
import backbone.custom_metrics.Intrinsic_dimension as id_module
import backbone.data_handle.Test as test
import backbone.visuals.VISUAL as viz

REP_DIR = "/idia/projects/camil/Koketso/galaxy_zoo_representations"
PLOTS_DIR = "../plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

ModuleNotFoundError: No module named 'backbone.AstroMLmodified'

## Load representations

Each model has an unlabeled set (`*_reps.h5`, used for the 2PCF/ID scores) and a
labeled classification-validation set (`*_reps_c.h5`, used for clustering
accuracy and Davies-Bouldin).

In [ ]:
def open_embeds(file_path):
    #load an embeddings/labels/ids triple saved by gz_foundation_models.py's h5 test runs
    with h5py.File(file_path, "r") as f:
        rep = np.array(f["embeddings"]).astype(np.float32)
        labels = np.array(f["labels"]).astype(str).tolist()
        ids = np.array(f["ids"]).astype(str).tolist()
    return rep, labels, ids


def discover_models(rep_dir):
    #pair each "<name>_reps.h5" with its labeled "<name>_reps_c.h5" counterpart
    models = {}
    for path in sorted(glob.glob(os.path.join(rep_dir, "*_reps.h5"))):
        name = os.path.basename(path)[: -len("_reps.h5")]
        class_path = os.path.join(rep_dir, f"{name}_reps_c.h5")
        if os.path.exists(class_path):
            models[name] = (path, class_path)
    return models


model_paths = discover_models(REP_DIR)
model_paths

In [4]:
representations = {}
for name, (rep_path, class_path) in model_paths.items():
    rep, label, ids = open_embeds(rep_path)
    rep_c, label_c, ids_c = open_embeds(class_path)
    representations[name] = {
        "rep": rep, "label": label, "ids": ids,
        "rep_c": rep_c, "label_c": label_c, "ids_c": ids_c,
    }
    print(f"{name}: rep={rep.shape}, rep_c={rep_c.shape}")

dinov3_convnext_base: rep=(269760, 1024), rep_c=(7315, 1024)
dinov3_vith: rep=(269760, 1280), rep_c=(7315, 1280)
imnet_convnext_base: rep=(269760, 1024), rep_c=(7315, 1024)
imnet_resnet18: rep=(269760, 512), rep_c=(7315, 512)
zoobot: rep=(269760, 1024), rep_c=(7315, 1024)


## Compute quality metrics

For each model: 2PCF score and ID score on the unlabeled set, clustering
accuracy and Davies-Bouldin score on the labeled classification set (the
latter computed on a 15-component PCA projection, matching `galaxy_zoo_tpcfs.py`).

In [ ]:
rows = []
for name, d in representations.items():
    rep = d["rep"]
    rep_c = d["rep_c"]
    labels_c = d["label_c"]

    tpcf_mean, tpcf_std = AstroMLmod.TPCF_score(rep)
    id_mean, id_std = id_module.id_score(rep)
    clustering_accuracy = test.clustering_accuracy((rep, d["ids"]), (labels_c, d["ids_c"]))
    davies = davies_bouldin_score(viz.pca(rep_c, n_components=15, verbose=False), labels_c)
    knn_accuracy, knn_std = test.KNN_accuracy(rep_c, labels_c)

    cosine_matrix = cosine_similarity(rep)
    off_diagonal = ~np.eye(len(rep), dtype=bool)
    cosine_values = cosine_matrix[off_diagonal]
    centered = rep - rep.mean(axis=0, keepdims=True)
    singular_values = np.linalg.svd(centered, compute_uv=False)
    anisotropy = (singular_values[0] ** 2) / np.square(singular_values).sum()

    rows.append({
        "model": name,
        "tpcf_mean": tpcf_mean,
        "tpcf_std": tpcf_std,
        "id_mean": float(np.ravel(id_mean)[0]),
        "id_std": float(np.ravel(id_std)[0]),
        "avg_cosine_similarity": cosine_values.mean(),
        "cosine_similarity_std": cosine_values.std(),
        "anisotropy": anisotropy,
        "clustering_accuracy": clustering_accuracy,
        "davies_bouldin": davies,
        "knn_accuracy": knn_accuracy,
        "knn_accuracy_std": knn_std,
    })

scores = pd.DataFrame(rows)
scores.to_csv(os.path.join(PLOTS_DIR, "deep_feature_quality_scores.csv"), index=False)
scores

Length subset used:  7268
Clustering accuracy 0.687809576224546
dinov3_convnext_base: TPCF=43777.0, ID=19.219355257724086, accuracy=0.687809576224546, davies=1.864
Length subset used:  7268
Clustering accuracy 0.8509906439185471
dinov3_vith: TPCF=43777.0, ID=20.181405612925808, accuracy=0.8509906439185471, davies=1.537
Length subset used:  7268
Clustering accuracy 0.8383324160704458
imnet_convnext_base: TPCF=43777.0, ID=22.03472732260439, accuracy=0.8383324160704458, davies=2.411
Length subset used:  7268
Clustering accuracy 0.712988442487617
imnet_resnet18: TPCF=43777.0, ID=24.61608248827357, accuracy=0.712988442487617, davies=2.152
Length subset used:  7268
Clustering accuracy 0.9828013208585581
zoobot: TPCF=43777.0, ID=15.25303166390295, accuracy=0.9828013208585581, davies=1.032


,model,tpcf_mean,tpcf_std,id_mean,id_std,clustering_accuracy,davies_bouldin
0,dinov3_convnext_base,43777.0,169.54,19.219355,0.028274,0.687810,1.863941
1,dinov3_vith,43777.0,169.54,20.181406,0.084270,0.850991,1.537016
2,imnet_convnext_base,43777.0,169.54,22.034727,0.109223,0.838332,2.410780
3,imnet_resnet18,43777.0,169.54,24.616082,0.075870,0.712988,2.152048
4,zoobot,43777.0,169.54,15.253032,0.049556,0.982801,1.032398


## Plot every metric against every other metric

Paper-quality (300 dpi, serif font) scatter plots, one PNG per metric pair,
saved to `../plots`.

In [ ]:
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linewidth": 0.5,
    "lines.linewidth": 1.5,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.top": True,
    "ytick.right": True,
    "legend.frameon": False,
})

ACCENT = "#1f4e79"
METRICS = [
    ("tpcf_mean", "2PCF score", "tpcf_std", "clustering_accuracy", "Clustering accuracy"),
    ("id_mean", "Intrinsic dimension", "id_std", "clustering_accuracy", "Clustering accuracy"),
    ("avg_cosine_similarity", "Avg. cosine similarity", "cosine_similarity_std", "clustering_accuracy", "Clustering accuracy"),
    ("cosine_similarity_std", "Cosine similarity std.", None, "clustering_accuracy", "Clustering accuracy"),
    ("anisotropy", "Anisotropy (top-1 EVR)", None, "clustering_accuracy", "Clustering accuracy"),
    ("davies_bouldin", "Davies-Bouldin score", None, "knn_accuracy", "KNN accuracy (%)"),
    ("knn_accuracy", "KNN accuracy (%)", "knn_accuracy_std", "clustering_accuracy", "Clustering accuracy"),
]


def plot_metric_against_accuracy(ax, data, metric, metric_label, error_column, accuracy, accuracy_label, panel_label):
    metric_error = data[error_column] if error_column else None
    ax.errorbar(data[accuracy], data[metric], yerr=metric_error, fmt="o", color=ACCENT, ecolor="gray", capsize=3, markersize=6, zorder=2)
    for _, row in data.iterrows():
        ax.annotate(row["model"], (row[accuracy], row[metric]), textcoords="offset points", xytext=(5, 4), fontsize=8)
    ax.set_xlabel(accuracy_label)
    ax.set_ylabel(metric_label)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.text(-0.15, 1.08, f"({panel_label})", transform=ax.transAxes, fontsize=11, fontweight="bold")


n_cols = 3
n_rows = -(-len(METRICS) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.3 * n_cols, 2.8 * n_rows), squeeze=False, constrained_layout=True)
axes = axes.flatten()

for panel_label, ax, metric_spec in zip(string.ascii_lowercase, axes, METRICS):
    plot_metric_against_accuracy(ax, scores, *metric_spec, panel_label)

for ax in axes[len(METRICS):]:
    ax.axis("off")

fig.savefig(os.path.join(PLOTS_DIR, "deep_feature_metrics_vs_accuracy.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(PLOTS_DIR, "deep_feature_metrics_vs_accuracy.png"), bbox_inches="tight")
plt.show()

In [ ]:
print(f"Saved metric table: {os.path.join(PLOTS_DIR, 'deep_feature_quality_scores.csv')}")
print(f"Saved plots: {os.path.join(PLOTS_DIR, 'deep_feature_metrics_vs_accuracy.pdf')} and .png")